# AI 기반 문장 생성 기술 분석 보고서


## 1. 학생 정보


- 훈련생 고유번호 : P172

- 성명 : 홍은비



---

# 2. 실습 파일 분석


# Practice_0 분석


## 2-1. 파일 목적 및 역할


`Attention Is All You Need` 논문을 기반으로 질문에 답하는 RAG 질의응답 파이프라인을 구현한다.
일반 LLM은 학습하지 않은 문서에 정확히 답하기 어렵고, 긴 논문 전체를 한 번에 프롬프트에 넣기에도 한계가 있다. 이를 해결하기 위해 논문을 작은 단위로 나누고, 질문과 관련된 부분만 검색해 GPT-4o에 근거로 제공한다. 또한 CoT → Self-Consistency → ReAct 관점으로 작성된 세 가지 복합 프롬프트를 실행한다. 이 과정에서 프롬프트 설계 기법이 답변 품질에 미치는 영향을 확인한다.


---


## 2-2. 적용 기술 분석


**[AI 기술]**
- RAG (Retrieval-Augmented Generation) : 대규모 언어 모델의 출력을 최적화하여 응답을 생성하기 전에 훈련 데이터 소스 외부의 신뢰할 수 있는 기술 자료를 참조하도록 하는 프로세스
- 벡터 임베딩 기반 의미 검색 : 단어의 단순 일치가 아닌 의미 유사도로 문서를 검색

**[모델]**
| 구분 | 모델 | 역할 |
|---|---|---|
| 임베딩 | `text-embedding-3-small` (OpenAI) | 텍스트 청크를 벡터로 변환 |
| 생성 | `gpt-4o` (OpenAI) | 검색된 근거를 바탕으로 답변 생성 |

**[라이브러리]**
| 구분 | 기술 | 역할 |
|---|---|---|
| 문서 처리 | `pypdf` | PDF 페이지의 텍스트 추출 |
| 청크 분할 | `RecursiveCharacterTextSplitter` | 500자 단위, 50자 중첩으로 문맥 보존 |
| 임베딩 | `text-embedding-3-small` | 청크를 의미 벡터로 변환 |
| 검색 | `FAISS` | 질문과 유사한 청크 상위 3개 검색 |
| 생성 | `ChatOpenAI(gpt-4o, temperature=0.3)` | 검색 근거를 바탕으로 답변 생성 |
| 환경 설정 | `python-dotenv` | API Key를 환경변수에서 안전하게 로드 |


**[데이터 처리 방법]**
1. PDF 전체 페이지를 순회하며 텍스트를 추출하고 개행으로 연결
2. `chunk_size=500`, `chunk_overlap=50` 설정으로 분할 → 총 89개 청크 생성
   - overlap 50자를 두는 이유는 청크 경계에서 문맥이 끊기는 현상을 완화하기 위함
3. 각 청크를 임베딩 벡터로 변환해 FAISS 인덱스에 저장 (저장 문서 수 89건)
4. 질의 시 로 상위 3개 청크를 검색해 프롬프트에 주입

---


## 2-3. 전체 동작 과정


**인덱싱:**: PDF 로드 → 텍스트 추출 → 청크 분할 → 임베딩 → FAISS 저장

**질의응답:**: 질문 입력 → 유사 청크 Top-3 검색 → 근거 문맥 구성 → 질문과 함께 GPT-4o에 전달 → 답변 출력




---


## 2-4. 주요 코드 분석



### 주요 코드



```python
# [1] 청크 분할 및 벡터 DB 생성
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,     # 하나의 청크가 가질 최대 문자 수
    chunk_overlap=50,   # 이전 청크와 공유할 문자 수
)
docs = text_splitter.create_documents([paper_text])   # 89개 청크

embedding_model = OpenAIEmbeddings(
    api_key=openai_api_key,
    model="text-embedding-3-small",
)
vectorstore = FAISS.from_documents(docs, embedding_model)
```

**역할** 긴 논문을 검색 가능한 청크로 나누고 임베딩하여 벡터 DB를 만든다. 




```python
# [2] 검색 + 생성 (RAG의 핵심)
def answer_with_context(prompt, top_k=3):
    related_docs = vectorstore.similarity_search(prompt, k=top_k)
    context = "\n\n".join(doc.page_content for doc in related_docs)
    full_prompt = f"""너는 논문 기반 연구 보조 AI야.
다음은 논문에서 발췌한 내용이야:

{context}

아래 질문에 답변해줘:
{prompt}"""
    llm = ChatOpenAI(model="gpt-4o", temperature=0.3)
    return llm.invoke([HumanMessage(content=full_prompt)]).content
```

**역할**: 질문과 유사한 근거를 검색한 뒤 LLM 입력에 포함하는 Retrieval + Generation의 핵심 코드다. 다만 현재 코드는 `invoke()`를 한 번만 호출하므로, Self-Consistency의 다중 응답 비교나 ReAct의 도구 호출 반복은 프롬프트 문구로만 표현되어 있고 실행 메커니즘으로는 구현되어 있지 않다.



# Practice_1-1 분석


## 3-1. 파일 목적 및 역할


사전학습된 LLM을 API로 호출하면 내부 동작이 블랙박스로 남는다. 이에 언어 모델이 어떤 원리로 다음 글자를 예측하는지, 그리고 순환 신경망 계열 모델이 왜 긴 문장에서 실패하는지 확인하려 한다.

이에 PyTorch로 문자 단위 LSTM 언어 모델을 직접 학습하여 다음 문자 예측과 자기회귀 문장 생성 원리를 확인한다.

학습 문장 `"hello world machine learning is fun "`에서 길이 10의 입력 시퀀스와 한 글자 뒤의 정답 시퀀스를 만든다. 200 epoch 학습 후 `"hello"`를 시작으로 50자를 생성한다. 작은 예제를 통해 LSTM의 동작과 순환 구조의 장기 의존성 한계를 관찰할 수 있다.


---


## 3-2. 적용 기술 분석


**모델 구조:** `Embedding → LSTM(2단) → Linear`

| 구성 요소 | 설정 | 역할 |
|---|---|---|
| Embedding | 문자 ID → 50차원 | 정수 문자를 학습 가능한 벡터로 변환 |
| LSTM | hidden 50, 2 layers | 이전 문맥을 hidden/cell state에 저장 |
| Linear | 50 → 문자 종류 수 | 다음 문자별 logit 출력 |
| 손실 함수 | CrossEntropyLoss | 예측 문자와 정답 문자 차이 계산 |
| 최적화 | Adam, lr=0.01 | 역전파 결과로 가중치 갱신 |

생성은 매 단계 `argmax`로 가장 높은 점수의 문자를 선택하는 Greedy Decoding이며, 생성된 문자를 다시 입력하는 자기회귀 방식이다. 따라서 같은 입력에는 같은 결과가 나오지만 특정 패턴이 반복되기 쉽다.


---


## 3-3. 데이터 처리 과정 설명


입력 데이터가 처리되어
최종 문장이 생성되는 과정을 단계별로 설명하세요.

1. 원문에서 고유 문자 목록 생성
2. `char_to_idx`, `idx_to_char`로 문자와 ID 매핑
3. 슬라이딩 윈도우로 입력 10자와 한 칸 뒤 정답 10자 구성
4. `LongTensor`로 PyTorch가 연산 가능한 형태로 전환
5. 순전파 → 손실 계산 → 역전파 → 가중치 갱신
6. `"hello"` 입력 후 다음 문자를 50회 반복 예측

즉, `hello worl`을 입력했을 때 `ello world`를 정답으로 두어 모든 위치에서 다음 문자 예측을 학습한다.





---


## 3-4. 모델 구조 설명


사용된 모델의 내부 구조와
각 구성 요소의 역할을 설명하세요.


- **Embedding Layer:** 희소한 문자 ID를 밀집 벡터로 표현한다.
- **LSTM Layer:** 입력/망각/출력 게이트로 저장할 정보와 버릴 정보를 조절한다. cell state는 장기 기억, hidden state는 현재 문맥의 요약 역할을 한다.
- **Linear Layer:** 각 시점의 50차원 표현을 전체 문자에 대한 점수로 바꾼다.

LSTM은 일반 RNN보다 장기 의존성을 잘 처리하지만 토큰을 순서대로 계산해야 하므로 병렬화가 어렵고, 긴 문맥을 제한된 상태에 압축하면서 초반 정보가 약해질 수 있다.


---


## 3-5. 주요 코드 분석


문장 생성과 관련된 핵심 코드를 작성하고
코드의 역할을 설명하세요.



### 주요 코드


```python

# [1] 학습 데이터 생성 - "다음 문자 예측" 과제 정의
for i in range(0, len(text) - seq_length):
    input_seq  = text[i : i+seq_length]        # 입력 문자열
    target_seq = text[i+1 : i+seq_length+1]    # 한 글자씩 앞으로 이동한 정답
    input_data.append([char_to_idx[c] for c in input_seq])
    target_data.append([char_to_idx[c] for c in target_seq])
```

**역할** : 입력보다 정답을 한 칸 뒤로 이동시켜 다음 문자 예측용 학습쌍을 만든다.


```python
# [2] 모델 정의
class CharLSTM(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers):
        super(CharLSTM, self).__init__()
        self.embedding = nn.Embedding(input_size, hidden_size)   # 문자 ID -> 벡터
        self.lstm = nn.LSTM(hidden_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, input_size)             # 다음 문자 예측

    def forward(self, x, hidden):
        x = self.embedding(x)              # 문자 임베딩
        out, hidden = self.lstm(x, hidden) # 순차 처리, 문맥 누적
        out = self.fc(out)                 # 어휘 크기만큼의 logit 출력
        return out, hidden
```

**역할** : 문자 ID를 임베딩하고, LSTM으로 문맥을 누적한 뒤 다음 문자 점수로 변환한다.



```python
# [3] 문장 생성 - 자기회귀 방식
def generate_text(start_str, length):
    model.eval()                                  # 추론 모드
    chars_input = torch.LongTensor([[char_to_idx[c] for c in start_str]])
    hidden = None
    result = start_str

    for _ in range(length):
        output, hidden = model(chars_input, hidden)
        last_char_logits = output[:, -1, :]                    # 마지막 시점 출력만 사용
        last_char_idx = torch.argmax(last_char_logits, dim=1).item()  # 최고 확률 문자
        result += idx_to_char[last_char_idx]
        chars_input = torch.LongTensor([[last_char_idx]])      # 생성 문자를 다음 입력으로

    return result

print(generate_text("hello", 50))
```

**역할** : 마지막 시점의 최고 점수 문자를 결과에 추가하고 다음 입력으로 재사용한다. 이 결정론적 선택은 재현성이 높지만 반복 문구를 만들 수 있다.



# Practice_1-2 분석


## 4-1. 파일 목적 및 역할


사전학습된 GPT-2로 문장을 생성해 LSTM보다 자연스러운 결과를 확인하고, 동시에 발생하는 환각(Hallucination)을 간단한 검색 결합으로 완화한다.

첫 번째 실습은 `"The capital of South Korea is"`를 GPT-2가 이어 쓰게 한다. 두 번째 실습은 국가별 수도가 담긴 미니 지식베이스에서 사실을 찾고 질문 앞에 붙여 생성한다. 이를 통해 LLM의 특징은 유창함이며 사실성을 확보하기 위해선 외부 근거가 필요하다는 점을 확인한다.




---


## 4-2. 적용 기술 분석



| 항목 | 내용 |
|---|---|
| 모델 | GPT-2 base, Decoder-only Transformer |
| 라이브러리 | Hugging Face `transformers`, PyTorch |
| 토큰화 | `GPT2Tokenizer`의 BPE 기반 서브워드 토큰화 |
| 입력 | `input_ids`, `attention_mask` |
| 생성 | 자기회귀 샘플링, `temperature=0.7` |
| 검색 | 딕셔너리 key의 문자열 포함 여부 확인 |

GPT-2는 대규모 텍스트로 이미 사전학습되어 있어 별도 학습 없이 사용할 수 있다. 다만 사전학습만 마친 Base 모델로 지시학습(Instruction Tuning)이나 RLHF를 거치지 않았으므로 질문에 정확히 답하기보다 입력 뒤에 올 법한 문장을 그럴듯하게 생성한다. 이는 Practice_0의 GPT-4o가 CoT 지시를 따르는 것과 대비되는 지점이며, 실행 결과에서 나타나는 환각의 배경이기도 하다.

라이브러리의 경우 별도의 RAG 프레임워크 없이 파이썬 기본 딕셔너리로 검색을 구현한 점이 Practice_0과의 차이다.


### 데이터 처리 방식

1. **토큰화:** 입력 문장을 `GPT2Tokenizer`로 BPE 서브워드 토큰 ID(`input_ids`)로 변환한다.
2. **마스크 생성:** 실제 입력 토큰과 패딩 영역을 구분하는 `attention_mask`를 함께 만든다.
3. **모델 입력:** 두 텐서를 GPT-2에 전달해 마지막 위치의 다음 토큰 확률 분포를 계산한다.
4. **반복 생성:** 분포에서 토큰을 샘플링하고, 생성된 토큰을 기존 입력 뒤에 붙여 `max_length`까지 자기회귀적으로 반복한다.
5. **디코딩:** 생성된 토큰 ID를 문자열로 복원하며 특수 토큰은 제거한다.
6. **검색 결합:** 두 번째 실습에서는 질문에 국가명이 포함되어 있는지 소문자 기준으로 검사하고, 일치한 사실 문장을 질문 앞에 추가한 뒤 동일한 생성 과정을 수행한다.

Practice_0이 문서를 청크로 나누고 임베딩/FAISS로 의미 검색하는 것과 달리, 이 파일은 소규모 딕셔너리의 문자열 부분 일치만 사용한다. 구현은 단순하지만 표현이 달라지거나 지식베이스에 없는 질문에는 검색이 실패한다.


### 문장 생성 방식

`model.generate()`를 사용한 자기회귀 생성이다.

`do_sample=True`는 확률 분포에서 토큰을 샘플링하고, `temperature=0.7`은 분포를 비교적 선명하게 만들어 무작위성을 조절한다. 이로 인해 실행할 때마다 결과가 달라지는 대신 사실과 다른 토큰이 선택될 수 있다.


---


## 4-3. 입력부터 결과 생성 과정


사용자의 입력이 최종 문장으로 생성되는
전체 과정을 단계별로 설명하세요.


**기본 생성:** 프롬프트 → BPE 토큰화 → GPT-2 → 다음 토큰 샘플링 반복 → 디코딩

**검색 결합 생성:** 질문 → 국가명 문자열 검색 → 사실 문장 반환 → 사실+질문 결합 → GPT-2 생성 → 디코딩

검색된 사실을 질문보다 앞에 배치하면 뒤쪽 토큰이 앞선 근거를 참조할 수 있어 환각을 줄이는 데 도움이 된다. 그러나 키워드가 정확히 포함되어야만 검색되므로 의미가 같은 다른 표현에는 취약하다.



---


## 4-4. 모델 구조 설명


GPT-2는 Transformer의 디코더만 사용하는(Decoder-only) 구조다.

핵심 구조는 다음과 같다.

1. **Token Embedding:** 토큰 ID를 벡터로 변환
2. **Position Embedding:** 토큰 순서 정보 추가
3. **Masked Multi-Head Self-Attention:** 현재 위치보다 뒤의 토큰은 가리고, 앞선 문맥의 여러 관계를 병렬로 계산
4. **Feed-Forward Network:** 각 위치의 표현을 비선형 변환
5. **Residual Connection + LayerNorm:** 정보와 기울기 흐름 안정화
6. **LM Head:** 다음 토큰의 어휘별 logit 출력


### LSTM과의 핵심 차이

| 항목 | LSTM | GPT-2 |
|---|---|---|
| 문맥 처리 | hidden state에 순차 압축 | Attention으로 이전 토큰 직접 참조 |
| 학습 병렬화 | 어려움 | 학습 시 행렬 연산 가능 |
| 생성 단위 | 문자 | BPE 서브워드 토큰 |
| 모델 준비 | 소규모 데이터로 직접 학습 | 대규모 사전학습 모델 활용 |

Transformer도 생성 단계에서는 토큰을 하나씩 만들지만, 학습 시 전체 시퀀스를 병렬 처리할 수 있다는 점이 중요하다.

---


## 4-5. 주요 코드 분석


### 주요 코드


```python
# [1] 문장 생성 함수
def generate_text(prompt, max_length=50):
    inputs = tokenizer(prompt, return_tensors="pt")
    input_ids = inputs["input_ids"]
    attention_mask = inputs["attention_mask"]

    outputs = model.generate(
        input_ids,
        attention_mask=attention_mask,
        max_length=max_length,
        do_sample=True,          # 확률 분포에서 샘플링
        temperature=0.7,         # 분포의 뾰족함 조절
        pad_token_id=tokenizer.eos_token_id,  # GPT2는 pad_token이 없어 eos로 대체
    )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)
```

**역할** : 프롬프트를 텐서로 변환하고 확률적으로 다음 토큰을 생성한 뒤 문자열로 복원한다. GPT-2에는 기본 pad token이 없어 eos token을 대신 사용한다.


```python
# [2] 미니 지식베이스와 검색 함수
knowledge_base = {
    "South Korea": "The capital of South Korea is Seoul.",
    "France": "The capital of France is Paris.",
    "Italy": "The capital of Italy is Rome.",
}

def retrieve_fact(query):
    for key in knowledge_base:
        if key.lower() in query.lower():
            return knowledge_base[key]
    return "I don't know."
```

**역할** : RAG의 Retrieval 단계를 가장 단순한 형태로 구현한 것이다. 다만 임베딩이나 유사도 계산 없이 문자열 부분 일치만 검사하므로, 표현이 조금만 달라져도 검색에 실패한다. Practice_0의 FAISS 벡터 검색과 비교하면 이 부분이 명확한 개선 대상이다.

```python
# [3] 검색 결과를 프롬프트에 주입
def generate_with_retrieval(query):
    fact = retrieve_fact(query)
    prompt = fact + " " + query      # 사실을 질문 앞에 배치
    return generate_text(prompt, 50)

query = "What is the capital of South Korea?"
print(generate_with_retrieval(query))
```

**역할** : 검색된 사실 문장을 질문 앞에 배치함으로써, 모델이 다음 토큰을 예측할 때 그 사실이 문맥으로 작용하게 만든다. 



# 5. 세 파일 기술 비교 분석


## 5-1. 파일별 기술 비교



| 비교 항목 | Practice_0 | Practice_1-1 | Practice_1-2 |
|---|---|---|---|
| 주요 목적 | 논문 근거 기반 질의응답 | 다음 문자 예측 원리 학습 | GPT-2 생성과 환각 완화 |
| 핵심 기술 | GPT-4o + 임베딩 + FAISS | PyTorch 2층 LSTM | 사전학습 GPT-2 Transformer |
| 데이터 처리 | PDF 청크 → 임베딩 → 의미 검색 | 문자 ID → 길이 10 시퀀스 | BPE 토큰화 + 문자열 검색 |
| 생성 방식 | 검색 근거를 넣어 LLM 호출 | Greedy 문자 생성 | 확률적 토큰 샘플링 |
| 장점 | 외부 문서 활용, 근거성 향상 | 구조가 단순하고 원리 확인 용이 | 별도 학습 없이 자연스러운 생성 |
| 한계 | API 비용/검색 품질 의존 | 작은 데이터, 반복, 장기 문맥 한계 | 환각, 단순 검색, 결과 변동 |
| 활용 분야 | 문서 QA/사내 검색 | 교육/간단 패턴 생성 | 초안/자동완성/챗봇 |




---


# 5-2. 기술 발전 과정 분석


기술 흐름은 **Practice_1-1 → Practice_1-2 → Practice_0**으로 볼 수 있다.

1. **LSTM 직접 학습:** hidden state로 이전 문맥을 전달하며 다음 문자를 생성한다. 원리는 명확하지만 작은 데이터에서는 암기와 반복이 심하고 긴 문맥 처리에 한계가 있다.
2. **사전학습 Transformer:** Self-Attention과 대규모 사전학습으로 문법과 문맥 품질이 크게 향상된다. 그러나 모델은 사실을 조회하는 시스템이 아니므로 유창하지만 틀린 답을 만들 수 있다.
3. **RAG 서비스:** 외부 문서를 검색해 근거로 제공함으로써 환각을 줄이고, 모델을 재학습하지 않아도 지식을 교체·갱신할 수 있다.

즉, 발전 방향은 생성 능력 향상에서 끝나지 않고 검색 근거와 검증 가능성 강화로 이어진다.



---


# 5-3. 각 방식의 개선 방향 비교



| 파일 | 우선 개선 방향 |
|---|---|
| Practice_0 | 출처·페이지 표시, 문단 단위 청크, 검색 결과 재순위화, 정답/근거 평가셋 구축 |
| Practice_1-1 | 데이터 확대, 검증셋 분리, 확률 샘플링·반복 억제, Attention 구조 검토 |
| Practice_1-2 | 문자열 검색을 임베딩 검색으로 교체, 지식베이스 확대, `top_p`·반복 패널티 적용, 근거 출력 |

Practice_0의 Self-Consistency와 ReAct는 현재 프롬프트 표현에 머물러 있다. 실제 적용하려면 여러 응답을 비교하는 절차와 Thought → Action → Observation 형태의 도구 호출 루프가 필요하다.



---


# 6. 실제 서비스 적용 방안



**① 검색 기반 질의응답 서비스 (Practice_0 방식)**

사내 규정/기술 문서/계약서를 벡터 DB로 구축해 직원이 자연어로 질문하면 근거 조항과 함께 답변하는 시스템. 문서만 교체하면 지식이 갱신되므로 모델 재학습이 필요 없고, 출처를 함께 제시하므로 답변의 신뢰성을 검증할 수 있다. 예) 신규 입사자 온보딩 Q&A, 고객센터 상담원 지원, 법무 계약서 1차 검토.

**② 교육 서비스 (Practice_1-1 방식)**

모델을 직접 학습시켜 보는 실습 환경. 손실이 줄어드는 과정과 생성 결과의 변화를 눈으로 확인하며 딥러닝 원리를 익히는 데 적합하다. 규모가 작아 GPU 없이도 실행 가능하다는 것이 교육용으로서의 장점이다.

**③ 콘텐츠 생성 서비스 (Practice_1-2 방식)**

사전학습 모델의 자연스러운 생성 능력을 활용해 초안을 만들고 사람이 검수하는 워크플로우. 예) 마케팅 카피 초안, 이메일 자동 완성, 코드 주석 생성. 환각 가능성이 있으므로 사실 확인이 필요한 영역에는 단독 적용하지 않는다.


---


# 7. 최종 의견

세 파일은 서로 경쟁하는 방식이 아니라 언어 생성 기술의 발전 단계를 보여준다. Practice_1-1은 다음 토큰 예측과 자기회귀 생성의 기본 원리, Practice_1-2는 Self-Attention과 사전학습의 효과, Practice_0은 외부 근거를 연결하는 RAG의 필요성을 보여준다.

특히 GPT-2가 문법적으로 자연스럽지만 틀린 수도 정보를 생성한 사례는 LLM의 본질이 자연스러운 문장 생성에 있음을 보여준다. 특히 사실성을 보장하지 않는다는 점이 가장 큰 시사점이다. 사실을 프롬프트에 주입하면 정확도가 좋아지지만, 검색이 실패하면 생성 품질도 함께 낮아진다. 따라서 실제 AI 서비스의 핵심은 모델 크기만이 아니라 검색 품질, 출처 제시, 반복 억제, 정량 평가와 사람의 검증을 함께 갖추는 것임을 알 수 있다.



---


# 8. 제출 전 확인


아래 항목을 확인하세요.



|확인 항목|확인|
|-|-|
|세 파일 모두 분석 완료|O|
|주요 코드 작성 완료|O|
|코드 역할 설명 완료|O|
|기술 특징 작성 완료|O|
|장점과 한계 작성 완료|O|
|개선 방향 작성 완료|O|
|파일별 비교 작성 완료|O|
|최종 의견 작성 완료|O|
